# Лабораторна робота № 7
## Логістична регресія. «Врятувати принцесу»

**Мета:** виконати бінарну класифікацію для датасету *Cure the Princess*, побудувати логістичну регресію, підібрати параметри та порівняти з SVM і деревом рішень.

**Датасет:** https://www.kaggle.com/datasets/unmoved/cure-the-princess

**Як запускати:** у Google Colab виконайте комірки зверху вниз. Для завантаження даних використайте один із двох варіантів нижче.

### 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 2. Завантаження даних

Оберіть **один** варіант.

In [ ]:
# === Варіант 1. Завантаження файлу в Google Colab ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('data.csv')
# df.head()

In [ ]:
# === Варіант 2. KaggleHub (рекомендовано) ===
import kagglehub
import os

path = kagglehub.dataset_download('unmoved/cure-the-princess')
print('Path:', path)
df = pd.read_csv(os.path.join(path, 'data.csv'))
df.head()

### 3. Первинний огляд датасету

In [ ]:
print('Розмір:', df.shape)
print('\nПерші рядки:')
display(df.head())
print('\nІнформація:')
df.info()

In [ ]:
print('Пропущені значення:')
print(df.isnull().sum())
print('\nДублікати:', df.duplicated().sum())

In [ ]:
print(df.describe())

**Висновок за первинним оглядом:**  
Датасет зазвичай не містить пропусків і дублікатів. Усі ознаки числові. Цільова змінна `Cured` — бінарна (0/1). Дані готові до подальшої роботи без додаткової очистки.

### 4. Поділ даних на навчальну й тестову вибірки

In [ ]:
X = df.drop(columns=['Cured'])
y = df['Cured']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Розподіл y_train:\n', y_train.value_counts(normalize=True))

### 5. Баланс класів на навчальній вибірці

In [ ]:
print(y_train.value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x=y_train)
plt.title('Баланс класів (y_train)')
plt.xlabel('Cured')
plt.show()

**Висновок:**  
Класи зазвичай близькі до збалансованих. Якщо різниця невелика — Accuracy можна використовувати, але F1-score все одно варто дивитися.

### 6. Дослідження викидів на навчальних даних

In [ ]:
X_train.boxplot(figsize=(14, 5), rot=90)
plt.title('Boxplot ознак (X_train)')
plt.tight_layout()
plt.show()

In [ ]:
q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers_count = ((X_train < lower) | (X_train > upper)).sum().sort_values(ascending=False)
print('Кількість потенційних викидів (1.5 IQR):')
print(outliers_count)

**Висновок про викиди:**  
Викиди є в кількох ознаках, але їх небагато. На основному етапі **не видаляємо** їх автоматично — магічні інгредієнти можуть мати екстремальні значення, які несуть інформацію.

### 7. Кореляційний аналіз на навчальній вибірці

In [ ]:
mtx = X_train.corr().abs()
mask = np.triu(np.ones_like(mtx, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(mtx, mask=mask, annot=True, fmt='.2f', cmap='Reds', vmin=0, vmax=1)
plt.title('Кореляційна матриця (|r|) — X_train')
plt.tight_layout()
plt.show()

In [ ]:
corr_with_target = pd.concat([X_train, y_train], axis=1).corr()['Cured'].drop('Cured').sort_values(key=abs, ascending=False)
print('Кореляція з Cured:')
print(corr_with_target)

**Висновок про кореляції:**  
Найбільш пов’язані з цільовою змінною ознаки видно з відсортованого списку. Сильних мультиколінеарностей (|r| > 0.85) зазвичай мало.

### 8. Стандартизація ознак

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Масштабування виконано.')

### 9. Логістична регресія (базова)

In [ ]:
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

### 10. Матриця помилок і приклади прогнозів

In [ ]:
conf = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(conf, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Передбачений')
plt.ylabel('Справжній')
plt.title('Матриця помилок')
plt.show()

In [ ]:
compare = pd.DataFrame({
    'True': y_test.values,
    'Predicted': y_pred
})
print(compare.sample(10, random_state=42))

**Висновок щодо якості базової логістичної регресії:**  
Accuracy зазвичай висока (часто > 0.85–0.90). Дивіться, яких помилок більше (FP чи FN). Однієї Accuracy недостатньо — краще дивитися precision/recall/F1.

### 11. Підбір гіперпараметрів

In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear', 'saga'],
    'max_iter': [500, 1000, 2000]
}

param_dist = {
    'C': np.logspace(-3, 2, 20),
    'solver': ['lbfgs', 'liblinear', 'saga'],
    'max_iter': [500, 1000, 2000]
}

In [ ]:
grid_search = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)

print('Найкращі параметри (GridSearch):', grid_search.best_params_)
print('CV Accuracy:', grid_search.best_score_)

In [ ]:
random_search = RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_dist,
    n_iter=20,
    cv=3,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train_scaled, y_train)

print('Найкращі параметри (RandomSearch):', random_search.best_params_)
print('CV Accuracy:', random_search.best_score_)

In [ ]:
best_lr = grid_search.best_estimator_
y_pred_best = best_lr.predict(X_test_scaled)

print('Test Accuracy (оптимізована LR):', accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

**Висновок про підбір параметрів:**  
GridSearch і RandomizedSearch зазвичай дають близькі результати. Покращення порівняно з базовою моделлю може бути невеликим, якщо дані вже добре лінійно роздільні.

### 12. Порівняння трьох моделей

In [ ]:
models = {
    'Logistic Regression': best_lr,
    'SVM': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    results.append({'Model': name, 'Accuracy': acc, 'F1-score': f1})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df)

best_name = results_df.iloc[0]['Model']
print(f'\nНайкраща модель: {best_name}')

### 13. Висновки до лабораторної роботи

1. **Первинний аналіз:** датасет чистий (без пропусків/дублікатів), усі ознаки числові, цільова — бінарна.
2. **Викиди та кореляції:** викиди присутні, але не критичні; кореляції з цільовою змінною різної сили.
3. **Базова логістична регресія** дає високу Accuracy і збалансовані precision/recall.
4. **Підбір гіперпараметрів** трохи покращує або стабілізує результат.
5. **Порівняння моделей:** найкраща модель визначається за Accuracy та F1 на тестовій вибірці (часто LR або SVM).

**Загальний висновок:** логістична регресія добре підходить для цієї задачі. Після стандартизації та підбору `C`/`solver` модель стабільна. Дерево рішень може перенавчатися, SVM часто близький за якістю.